<a href="https://colab.research.google.com/github/Doumbia07/DI_Bootcamp/blob/main/daylichalleng5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================
# Daily Challenge: Classifying Handwritten Digits with CNNs
# ============================================

# 1. IMPORTS NÉCESSAIRES (version corrigée)
# ============================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPool2D
from tensorflow.keras.utils import to_categorical   # <--- Correction ici
import numpy as np

print("="*50)
print("CHARGEMENT ET EXPLORATION DES DONNÉES")
print("="*50)

# ============================================
# 2. CHARGER LE DATASET MNIST
# ============================================
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Forme de x_train :", x_train.shape)   # (60000, 28, 28)
print("Forme de y_train :", y_train.shape)   # (60000,)
print("Forme de x_test  :", x_test.shape)    # (10000, 28, 28)
print("Forme de y_test  :", y_test.shape)    # (10000,)
print("Nombre de classes :", len(np.unique(y_train))) # 10

print("\n" + "="*50)
print("MODÈLE 1 : RÉSEAU DE NEURONES FULLY CONNECTED (DENSE)")
print("="*50)

# ============================================
# 3. PRÉPROCESSEMENT POUR LE MODÈLE FULLY CONNECTED
# ============================================

# 3.a. Aplatir (Flatten) les images de 28x28 à 784 pixels
x_train_dense = x_train.reshape((x_train.shape[0], 28 * 28))
x_test_dense  = x_test.reshape((x_test.shape[0], 28 * 28))

# 3.b. Normaliser les valeurs des pixels en divisant par 255
x_train_dense = x_train_dense / 255.0
x_test_dense  = x_test_dense / 255.0

# 3.c. One-hot encoder les étiquettes avec to_categorical()
y_train_dense = to_categorical(y_train, 10)   # <--- remplace np_utils
y_test_dense  = to_categorical(y_test, 10)

print("x_train_dense shape :", x_train_dense.shape)  # (60000, 784)
print("y_train_dense shape :", y_train_dense.shape)  # (60000, 10)

# ============================================
# 4. CONSTRUIRE ET ENTRAÎNER LE MODÈLE FULLY CONNECTED
# ============================================

model_dense = Sequential()
model_dense.add(Dense(128, activation='relu', input_shape=(784,)))
model_dense.add(Dense(64, activation='relu'))
model_dense.add(Dense(10, activation='softmax'))

model_dense.compile(optimizer='adam',
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])

model_dense.summary()

print("\n--- Entraînement du modèle Fully Connected ---")
history_dense = model_dense.fit(x_train_dense, y_train_dense,
                                epochs=5,
                                batch_size=32,
                                validation_data=(x_test_dense, y_test_dense),
                                verbose=1)

test_loss_dense, test_acc_dense = model_dense.evaluate(x_test_dense, y_test_dense, verbose=0)
print(f"\n✅ Précision du modèle Fully Connected sur le test : {test_acc_dense:.4f} ({test_acc_dense*100:.2f}%)")

print("\n" + "="*50)
print("MODÈLE 2 : RÉSEAU DE NEURONES CONVOLUTIF (CNN)")
print("="*50)

# ============================================
# 5. PRÉPROCESSEMENT POUR LE MODÈLE CNN
# ============================================

# 5.a. Reshape : ajouter la dimension du canal (1 pour noir et blanc)
x_train_cnn = x_train.reshape((x_train.shape[0], 28, 28, 1))
x_test_cnn  = x_test.reshape((x_test.shape[0], 28, 28, 1))

# 5.b. Normaliser
x_train_cnn = x_train_cnn / 255.0
x_test_cnn  = x_test_cnn / 255.0

# 5.c. One-hot encoder (on réutilise la même fonction)
y_train_cnn = to_categorical(y_train, 10)
y_test_cnn  = to_categorical(y_test, 10)

print("x_train_cnn shape :", x_train_cnn.shape)  # (60000, 28, 28, 1)
print("y_train_cnn shape :", y_train_cnn.shape)  # (60000, 10)

# ============================================
# 6. CONSTRUIRE ET ENTRAÎNER LE MODÈLE CNN
# ============================================

model_cnn = Sequential()
model_cnn.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)))
model_cnn.add(MaxPool2D(pool_size=(2, 2)))
model_cnn.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model_cnn.add(MaxPool2D(pool_size=(2, 2)))
model_cnn.add(Flatten())
model_cnn.add(Dense(128, activation='relu'))
model_cnn.add(Dense(10, activation='softmax'))

model_cnn.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

model_cnn.summary()

print("\n--- Entraînement du modèle CNN ---")
history_cnn = model_cnn.fit(x_train_cnn, y_train_cnn,
                            epochs=5,
                            batch_size=32,
                            validation_data=(x_test_cnn, y_test_cnn),
                            verbose=1)

test_loss_cnn, test_acc_cnn = model_cnn.evaluate(x_test_cnn, y_test_cnn, verbose=0)
print(f"\n✅ Précision du modèle CNN sur le test : {test_acc_cnn:.4f} ({test_acc_cnn*100:.2f}%)")

print("\n" + "="*50)
print("7. COMPARAISON DES PERFORMANCES")
print("="*50)

print(f"Précision du modèle Fully Connected : {test_acc_dense:.4f}")
print(f"Précision du modèle CNN              : {test_acc_cnn:.4f}")
print(f"Différence (CNN - Dense)            : {test_acc_cnn - test_acc_dense:.4f}")

if test_acc_cnn > test_acc_dense:
    print("\n✅ Le modèle CNN est plus performant que le modèle Fully Connected.")
    print("   En effet, le CNN utilise la convolution pour capturer les motifs")
    print("   spatiaux (bords, courbes) présents dans les images de chiffres,")
    print("   tandis que le Dense traite chaque pixel indépendamment.")
else:
    print("\n⚠️ Les performances sont similaires, mais en général le CNN surpasse le Dense sur ce dataset.")

print("\n" + "="*50)
print("FIN DU CHALLENGE 🚀")
print("="*50)

CHARGEMENT ET EXPLORATION DES DONNÉES
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Forme de x_train : (60000, 28, 28)
Forme de y_train : (60000,)
Forme de x_test  : (10000, 28, 28)
Forme de y_test  : (10000,)
Nombre de classes : 10

MODÈLE 1 : RÉSEAU DE NEURONES FULLY CONNECTED (DENSE)
x_train_dense shape : (60000, 784)
y_train_dense shape : (60000, 10)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)


--- Entraînement du modèle CNN ---
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 66s 34ms/step - accuracy: 0.9617 - loss: 0.1262 - val_accuracy: 0.9866 - val_loss: 0.0400
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 65s 35ms/step - accuracy: 0.9871 - loss: 0.0419 - val_accuracy: 0.9895 - val_loss: 0.0336
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 61s 32ms/step - accuracy: 0.9911 - loss: 0.0289 - val_accuracy: 0.9874 - val_loss: 0.0402
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 88s 36ms/step - accuracy: 0.9936 - loss: 0.0209 - val_accuracy: 0.9889 - val_loss: 0.0335
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 79s 34ms/step - accuracy: 0.9948 - loss: 0.0161 - val_accuracy: 0.9901 - val_loss: 0.0303

✅ Précision du modèle CNN sur le test : 0.9901 (99.01%)

7. COMPARAISON DES PERFORMANCES
Précision du modèle Fully Connected : 0.9745
Précision du modèle CNN              : 0.9901
Différence (CNN - Dense)            : 0.0156

✅ Le modèle CNN est plus performant que le modèle Fully Connected.
   En effet, le